# RFM Customer Segmentation - Final Project

This project uses PySpark in Databricks to perform Recency, Frequency, and Monetary (RFM) customer segmentation on the Sales Data FY 2020–2021 dataset.

The analysis follows these main steps:

1. Load and inspect the sales data.
2. Check data quality and order-status values.
3. Retain qualifying `complete` and `paid` transactions.
4. Calculate Recency, Frequency, and Monetary values for each customer.
5. Assign RFM scores from 1 to 5.
6. Group customers into meaningful marketing segments.
7. Examine the four customers required by the assignment.
8. Prepare datasets and visualizations for the project report and presentation.

## Import the PySpark tools

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

## Connect to the source table: df_raw 

In [0]:
SOURCE_TABLE = "bia_5203.default.sales_06_fy_202021"
df_raw = spark.table(SOURCE_TABLE)

print(f"Rows: {df_raw.count():,}")
print(f"Columns: {len(df_raw.columns)}")
display(df_raw.limit(10))

Rows: 286,392
Columns: 36


order_id,order_date,status,item_id,sku,qty_ordered,price,value,discount_amount,total,category,payment_method,bi_st,cust_id,year,month,ref_num,Name Prefix,First Name,Middle Initial,Last Name,Gender,age,full_name,E Mail,Customer Since,SSN,Phone No.,Place Name,County,City,State,Zip,Region,User Name,Discount_Percent
100354678,2020-10-01,received,574772,oasis_Oasis-064-36,21,89.9,1798.0,0.0,1798.0,Men's Fashion,cod,Valid,60124,2020,Oct-20,987867,Drs.,Jani,W,Titus,F,43,"Titus, Jani",jani.titus@gmail.com,2006-08-22,627-31-5251,405-959-1129,Vinson,Harmon,Vinson,OK,73571,South,jwtitus,0.0
100354678,2020-10-01,received,574774,Fantastic_FT-48,11,19.0,190.0,0.0,190.0,Men's Fashion,cod,Valid,60124,2020,Oct-20,987867,Drs.,Jani,W,Titus,F,43,"Titus, Jani",jani.titus@gmail.com,2006-08-22,627-31-5251,405-959-1129,Vinson,Harmon,Vinson,OK,73571,South,jwtitus,0.0
100354680,2020-10-01,complete,574777,mdeal_DMC-610-8,9,149.9,1199.2,0.0,1199.2,Men's Fashion,cod,Net,60124,2020,Oct-20,987867,Drs.,Jani,W,Titus,F,43,"Titus, Jani",jani.titus@gmail.com,2006-08-22,627-31-5251,405-959-1129,Vinson,Harmon,Vinson,OK,73571,South,jwtitus,0.0
100354680,2020-10-01,complete,574779,oasis_Oasis-061-36,9,79.9,639.2,0.0,639.2,Men's Fashion,cod,Net,60124,2020,Oct-20,987867,Drs.,Jani,W,Titus,F,43,"Titus, Jani",jani.titus@gmail.com,2006-08-22,627-31-5251,405-959-1129,Vinson,Harmon,Vinson,OK,73571,South,jwtitus,0.0
100367357,2020-11-13,received,595185,MEFNAR59C38B6CA08CD,2,99.9,99.9,0.0,99.9,Men's Fashion,cod,Valid,60124,2020,Nov-20,987867,Drs.,Jani,W,Titus,F,43,"Titus, Jani",jani.titus@gmail.com,2006-08-22,627-31-5251,405-959-1129,Vinson,Harmon,Vinson,OK,73571,South,jwtitus,0.0
100367357,2020-11-13,received,595186,MEFBUY59B7C3DDC2CA3-42,2,39.9,39.9,0.0,39.9,Men's Fashion,cod,Valid,60124,2020,Nov-20,987867,Drs.,Jani,W,Titus,F,43,"Titus, Jani",jani.titus@gmail.com,2006-08-22,627-31-5251,405-959-1129,Vinson,Harmon,Vinson,OK,73571,South,jwtitus,0.0
100367360,2020-11-13,order_refunded,595192,MATDAN59C3C845B38F0,2,47.6,47.6,0.0,47.6,Mobiles & Tablets,cod,Valid,60124,2020,Nov-20,987867,Drs.,Jani,W,Titus,F,43,"Titus, Jani",jani.titus@gmail.com,2006-08-22,627-31-5251,405-959-1129,Vinson,Harmon,Vinson,OK,73571,South,jwtitus,0.0
100354677,2020-10-01,canceled,574769,GFE_19_USBLEDLight,2,49.0,49.0,0.0,49.0,Mobiles & Tablets,Payaxis,Gross,42485,2020,Oct-20,171143,Prof.,Lee,S,Eaker,M,28,"Eaker, Lee",lee.eaker@gmail.com,1981-02-04,185-86-4345,239-335-6755,Graham,Bradford,Graham,FL,32042,South,lseaker,0.0
100354677,2020-10-01,canceled,574770,oasis_Kingston-32GB-DTIG4,2,135.0,135.0,0.0,135.0,Computing,Payaxis,Gross,42485,2020,Oct-20,171143,Prof.,Lee,S,Eaker,M,28,"Eaker, Lee",lee.eaker@gmail.com,1981-02-04,185-86-4345,239-335-6755,Graham,Bradford,Graham,FL,32042,South,lseaker,0.0
100354677,2020-10-01,canceled,574771,Geepas_GSB5420,2,549.9,549.9,0.0,549.9,Appliances,Payaxis,Gross,42485,2020,Oct-20,171143,Prof.,Lee,S,Eaker,M,28,"Eaker, Lee",lee.eaker@gmail.com,1981-02-04,185-86-4345,239-335-6755,Graham,Bradford,Graham,FL,32042,South,lseaker,0.0


## Data-quality checks

Before calculating RFM values, we examine the dataset for:

- missing values;
- missing columns required for the analysis;
- the available order-status values; and
- duplicate rows.

These checks help confirm that the data is suitable for the RFM calculation and document the decisions made during preprocessing.

### Verify the required columns

In [0]:
required_columns = [
    "cust_id",
    "order_id",
    "order_date",
    "total",
    "status"
]

missing_required_columns = [
    column
    for column in required_columns
    if column not in df_raw.columns
]

assert not missing_required_columns, (
    f"Missing required columns: {missing_required_columns}"
)

print("All required RFM columns are available.")

All required RFM columns are available.


### Check for missing values

In [0]:
null_counts = df_raw.select([
    F.sum(
        F.when(F.col(f"`{column}`").isNull(), 1).otherwise(0)
    ).alias(column)
    for column in df_raw.columns
])

display(null_counts)

order_id,order_date,status,item_id,sku,qty_ordered,price,value,discount_amount,total,category,payment_method,bi_st,cust_id,year,month,ref_num,Name Prefix,First Name,Middle Initial,Last Name,Gender,age,full_name,E Mail,Customer Since,SSN,Phone No.,Place Name,County,City,State,Zip,Region,User Name,Discount_Percent
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


### Inspect the order statuses

In [0]:
status_counts = (
    df_raw
    .withColumn(
        "status_clean",
        F.lower(F.trim(F.col("status")))
    )
    .groupBy("status_clean")
    .count()
    .orderBy(F.desc("count"))
)

display(status_counts)

status_clean,count
canceled,112166
complete,88968
received,51775
order_refunded,25713
refund,3820
cod,2851
paid,756
closed,176
payment_review,57
pending,48


### Check for duplicate rows

In [0]:
duplicate_row_count = (
    df_raw.count() - df_raw.dropDuplicates().count()
)

print(f"Duplicate rows: {duplicate_row_count:,}")

Duplicate rows: 0


## Data preprocessing and analysis scope

The dataset contains several order statuses, including `complete`, `paid`, `received`, `canceled`, refunded, pending, and processing statuses.

For the primary RFM analysis, only transactions with a status of `complete` or `paid` are retained. This follows the data-transformation rule demonstrated in Lab 6 and provides a clear definition of a qualifying purchase.

All other statuses are excluded because they do not provide the same confirmation that the purchase was successfully completed. Including cancelled, refunded, pending, or otherwise uncertain transactions could overstate customer purchasing frequency and monetary value.

Rows with a missing customer ID, order ID, order date, or total are also excluded. Transactions with a total of zero or less are removed because they do not represent positive customer spending.

### Create the filtered analysis dataset: analysis_df 
> The filtering results are reported as an audit summary so that the number of retained and removed records, together with the reasons for removal, remain transparent. Of the 89,724 records marked `complete` or `paid`, 4,206 have a total of zero and are removed, leaving 85,518 qualifying transaction rows for the RFM analysis.

In [0]:
valid_statuses = ["complete", "paid"]

analysis_df = (
    df_raw
    .withColumn(
        "status_clean",
        F.lower(F.trim(F.col("status")))
    )
    .withColumn(
        "order_date",
        F.to_date(F.col("order_date"))
    )
    .filter(
        F.col("cust_id").isNotNull() &
        F.col("order_id").isNotNull() &
        F.col("order_date").isNotNull() &
        F.col("total").isNotNull() &
        F.col("status_clean").isin(valid_statuses) &
        (F.col("total") > 0)
    )
)

print(f"Qualifying rows: {analysis_df.count():,}")
print(
    f"Qualifying orders: "
    f"{analysis_df.select('order_id').distinct().count():,}"
)
print(
    f"Qualifying customers: "
    f"{analysis_df.select('cust_id').distinct().count():,}"
)

display(
    analysis_df
    .groupBy("status_clean")
    .count()
    .orderBy(F.desc("count"))
)

complete_paid_df = (
    df_raw
    .withColumn(
        "status_clean",
        F.lower(F.trim(F.col("status")))
    )
    .filter(
        F.col("status_clean").isin(valid_statuses)
    )
)

rows_before_cleaning = complete_paid_df.count()
rows_after_cleaning = analysis_df.count()
rows_removed = rows_before_cleaning - rows_after_cleaning

print(f"Complete/paid rows before cleaning: {rows_before_cleaning:,}")
print(f"Rows removed during cleaning: {rows_removed:,}")
print(f"Rows retained for RFM: {rows_after_cleaning:,}")

display(
    complete_paid_df
    .filter(
        F.col("total").isNull() |
        (F.col("total") <= 0)
    )
    .groupBy("status_clean")
    .count()
    .withColumnRenamed("count", "rows_removed")
    .orderBy(F.desc("rows_removed"))
)

Qualifying rows: 85,518
Qualifying orders: 61,258
Qualifying customers: 29,695


status_clean,count
complete,84795
paid,723


Complete/paid rows before cleaning: 89,724
Rows removed during cleaning: 4,206
Rows retained for RFM: 85,518


status_clean,rows_removed
complete,4173
paid,33


## Calculate customer-level RFM metrics

The filtered transaction data is aggregated so that each customer appears once in the RFM table.

The three RFM measurements are calculated as follows:

- **Recency:** Number of days between the latest qualifying transaction date in the dataset and the customer's most recent qualifying purchase. A lower value is better.
- **Frequency:** Number of distinct qualifying orders placed by the customer. A higher value is better.
- **Monetary:** Total amount spent by the customer across qualifying transactions. A higher value is better.

A common reference date is used for every customer so that recency values are comparable.

In [0]:
# Use the maximum date in the analysis data as the common recency reference date
reference_date = analysis_df.agg(F.max("order_date").alias("reference_date")).first()["reference_date"]
print("Reference date:", reference_date)

Reference date: 2021-09-30


### Calculate one RFM record per customer: rfm_base 

In [0]:
rfm_base = (
    analysis_df
    .groupBy("cust_id")
    .agg(
        F.max("order_date").alias("last_purchase_date"),
        F.countDistinct("order_id").alias("frequency"),
        F.round(F.sum("total"), 2).alias("monetary")
    )
    .withColumn(
        "recency",
        F.datediff(
            F.lit(reference_date),
            F.col("last_purchase_date")
        )
    )
    .select(
        "cust_id",
        "last_purchase_date",
        "recency",
        "frequency",
        "monetary"
    )
)

print(f"Customers in RFM analysis: {rfm_base.count():,}")

display(
    rfm_base
    .orderBy("cust_id")
    .limit(20)
)

Customers in RFM analysis: 29,695


cust_id,last_purchase_date,recency,frequency,monetary
4,2021-09-14,16,13,21079.95
15,2021-02-11,231,3,216.8
20,2021-04-01,182,5,22379.7
21,2021-02-03,239,1,105.0
28,2020-10-17,348,1,70.0
32,2021-05-28,125,62,44030.98
33,2021-05-26,127,36,29248.53
41,2020-11-18,316,1,219.9
44,2021-04-01,182,2,2237.02
47,2020-12-31,273,9,12425.01


### Examine the RFM distributions

In [0]:
display(
    rfm_base
    .select(
        "recency",
        "frequency",
        "monetary"
    )
    .summary()
)

summary,recency,frequency,monetary
count,29695,29695,29695
mean,234.91365549755852,2.0629062131672,2116.371786832586
stddev,62.90250672549147,3.719959449942177,5555.50320530421
min,0,1,0.1
25%,182,1,109.9
50%,263,1,302.5
75%,281,2,1698.4
max,364,325,227106.79


## Assign RFM scores

Each customer is assigned a score from 1 to 5 for Recency, Frequency, and Monetary value.

Customers are divided into five approximately equal-sized groups using `ntile(5)`:

- **Recency:** Lower values are better because they represent more recent purchases. Therefore, the recency ranking is reversed so that the most recent customers receive a score of 5.
- **Frequency:** Higher values are better, so customers with more distinct orders receive higher scores.
- **Monetary:** Higher values are better, so customers with greater total spending receive higher scores.

The three scores are combined as text to create an RFM score. For example, an RFM score of `545` represents a Recency score of 5, a Frequency score of 4, and a Monetary score of 5.

ntile(5) ranks customers relative to the rest of the customer population. It does not require the raw RFM values to be normalized first. Because `ntile(5)` creates approximately equal-sized groups, customers with the same raw value may occasionally fall into adjacent scoring groups.

### Calculate the R, F and M scores

In [0]:
recency_window = Window.orderBy(
    F.col("recency").asc(),
    F.col("cust_id").asc()
)

frequency_window = Window.orderBy(
    F.col("frequency").asc(),
    F.col("cust_id").asc()
)

monetary_window = Window.orderBy(
    F.col("monetary").asc(),
    F.col("cust_id").asc()
)

rfm = (
    rfm_base
    .withColumn(
        "R_Score",
        6 - F.ntile(5).over(recency_window)
    )
    .withColumn(
        "F_Score",
        F.ntile(5).over(frequency_window)
    )
    .withColumn(
        "M_Score",
        F.ntile(5).over(monetary_window)
    )
    .withColumn(
        "RFM_Score",
        F.concat(
            F.col("R_Score").cast("string"),
            F.col("F_Score").cast("string"),
            F.col("M_Score").cast("string")
        )
    )
)

display(
    rfm
    .orderBy("cust_id")
    .limit(20)
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


cust_id,last_purchase_date,recency,frequency,monetary,R_Score,F_Score,M_Score,RFM_Score
4,2021-09-14,16,13,21079.95,5,5,5,555
15,2021-02-11,231,3,216.8,3,5,3,353
20,2021-04-01,182,5,22379.7,4,5,5,455
21,2021-02-03,239,1,105.0,3,1,2,312
28,2020-10-17,348,1,70.0,1,1,1,111
32,2021-05-28,125,62,44030.98,5,5,5,555
33,2021-05-26,127,36,29248.53,5,5,5,555
41,2020-11-18,316,1,219.9,1,1,3,113
44,2021-04-01,182,2,2237.02,4,4,4,444
47,2020-12-31,273,9,12425.01,3,5,5,355


## Define Customer Segments: rfm 

The RFM scores are translated into five meaningful customer segments based on patterns described in the course material.

The following rules are applied:

- **Champions:** High Recency, Frequency, and Monetary scores (`R ≥ 4`, `F ≥ 4`, and `M ≥ 4`). These customers purchased recently, order frequently, and spend highly.
- **Loyal Customers:** Moderate-to-high Recency and Frequency scores (`R ≥ 3` and `F ≥ 3`). These customers demonstrate consistent purchasing behaviour.
- **Potential Loyalists:** High Recency score (`R ≥ 4`). These customers purchased recently but have not yet reached the frequency level of Loyal Customers or Champions.
- **Lost Customers:** Very low Recency and low Frequency scores (`R = 1` and `F ≤ 2`). These customers purchased infrequently and have not returned recently.
- **At Risk:** All remaining customers. These customers do not demonstrate sufficiently recent and frequent purchasing behaviour and may require retention or re-engagement efforts.

The rules are evaluated in order from the most specific segment to the broadest. This ensures that every customer is assigned to exactly one of the five named segments.

In [0]:

rfm = rfm.withColumn(
    "Segment",
    F.when(
        (F.col("R_Score") >= 4) &
        (F.col("F_Score") >= 4) &
        (F.col("M_Score") >= 4),
        "Champions"
    )
    .when(
        (F.col("R_Score") >= 3) &
        (F.col("F_Score") >= 3),
        "Loyal Customers"
    )
    .when(
        F.col("R_Score") >= 4,
        "Potential Loyalists"
    )
    .when(
        (F.col("R_Score") == 1) &
        (F.col("F_Score") <= 2),
        "Lost Customers"
    )
    .otherwise("At Risk")
)

display(
    rfm
    .orderBy("cust_id")
    .limit(20)
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


cust_id,last_purchase_date,recency,frequency,monetary,R_Score,F_Score,M_Score,RFM_Score,Segment
4,2021-09-14,16,13,21079.95,5,5,5,555,Champions
15,2021-02-11,231,3,216.8,3,5,3,353,Loyal Customers
20,2021-04-01,182,5,22379.7,4,5,5,455,Champions
21,2021-02-03,239,1,105.0,3,1,2,312,At Risk
28,2020-10-17,348,1,70.0,1,1,1,111,Lost Customers
32,2021-05-28,125,62,44030.98,5,5,5,555,Champions
33,2021-05-26,127,36,29248.53,5,5,5,555,Champions
41,2020-11-18,316,1,219.9,1,1,3,113,Lost Customers
44,2021-04-01,182,2,2237.02,4,4,4,444,Champions
47,2020-12-31,273,9,12425.01,3,5,5,355,Loyal Customers


## Summarize the customer segments

In [0]:
total_rfm_customers = rfm.count()

segment_summary = (
    rfm
    .groupBy("Segment")
    .agg(
        F.count("*").alias("customer_count"),
        F.round(F.avg("recency"), 2).alias("average_recency"),
        F.round(F.avg("frequency"), 2).alias("average_frequency"),
        F.round(F.sum("monetary"), 2).alias("total_monetary"),
        F.round(F.avg("monetary"), 2).alias("average_monetary")
    )
    .withColumn(
        "customer_percentage",
        F.round(
            F.col("customer_count") / F.lit(total_rfm_customers) * 100,
            2
        )
    )
    .select(
        "Segment",
        "customer_count",
        "customer_percentage",
        "average_recency",
        "average_frequency",
        "total_monetary",
        "average_monetary"
    )
    .orderBy(F.desc("customer_count"))
)

display(segment_summary)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Segment,customer_count,customer_percentage,average_recency,average_frequency,total_monetary,average_monetary
Loyal Customers,10874,36.62,195.4,1.65,1.373459686E7,1263.07
At Risk,9550,32.16,278.19,1.8,1.712253142E7,1792.94
Lost Customers,4762,16.04,307.67,1.0,2298107.44,482.59
Champions,3623,12.2,161.31,5.68,2.917678559E7,8053.21
Potential Loyalists,886,2.98,163.27,1.0,513638.9,579.73


## Required Customer RFM Results

The assignment specifically requests RFM segmentation results for customers `60149`, `2844`, `60767`, and `39707`.

The table below begins with all four required customer IDs and left-joins them to the completed/paid RFM results. This ensures that a customer is not silently omitted when no qualifying transaction exists.

Customer `60767` has no qualifying completed or paid transaction. Therefore, recency and the RFM scores cannot be calculated for this customer, and no RFM segment is assigned.

In [0]:
# Create a DataFrame containing all four assignment-required customers
required_customer_ids = spark.createDataFrame(
    [(60149,), (2844,), (60767,), (39707,)],
    ["cust_id"]
)

# Left join ensures that all four requested customer IDs remain in the table
required_customer_results = (
    required_customer_ids
    .join(
        rfm.select(
            "cust_id",
            "recency",
            "frequency",
            "monetary",
            "R_Score",
            "F_Score",
            "M_Score",
            "RFM_Score",
            "Segment"
        ),
        on="cust_id",
        how="left"
    )
    .withColumn(
        "Qualifying_Orders",
        F.coalesce(F.col("frequency"), F.lit(0))
    )
    .withColumn(
        "Qualifying_Revenue",
        F.coalesce(F.col("monetary"), F.lit(0.0))
    )
    .withColumn(
        "Result",
        F.when(
            F.col("Segment").isNull(),
            F.lit("No qualifying completed/paid transactions")
        ).otherwise(F.lit("Included in RFM analysis"))
    )
    .select(
        "cust_id",
        "Qualifying_Orders",
        F.round("Qualifying_Revenue", 2).alias("Qualifying_Revenue"),
        "recency",
        "R_Score",
        "F_Score",
        "M_Score",
        "RFM_Score",
        "Segment",
        "Result"
    )
    .orderBy("cust_id")
)

display(required_customer_results)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


cust_id,Qualifying_Orders,Qualifying_Revenue,recency,R_Score,F_Score,M_Score,RFM_Score,Segment,Result
2844,2,7147.8,138,5,4,5,545,Champions,Included in RFM analysis
39707,4,14493.6,142,5,5,5,555,Champions,Included in RFM analysis
60149,1,1999.9,364,1,1,4,114,Lost Customers,Included in RFM analysis
60767,0,0.0,null,null,null,null,null,null,No qualifying completed/paid transactions


## Dashboard preparation

A simple one-page dashboard is prepared to provide a rounded overview of the RFM results.

The proposed dashboard includes:

- KPI cards for qualifying customers, qualifying orders, total revenue, and average customer spending;
- global filters for year and customer segment;
- optional local filters for region or qualifying order status;
- customer count by segment;
- total revenue by segment;
- monthly qualifying-sales trend; and
- a customer-level RFM details table.

Two datasets are maintained at different levels of detail:

1. **Customer-level dataset:** Contains one row per customer and is used for customer counts, RFM values, segment analysis, and average customer spending.
2. **Transaction-level dataset:** Contains qualifying `complete` and `paid` transaction rows and is used for order counts, revenue, monthly trends, product categories, regions, and order-status analysis.

Keeping these datasets separate prevents customer monetary values from being counted repeatedly after joining them to transaction-level rows.

### Create the customer-level dashboard dataset: dashboard_customers_df

In [0]:
customer_profile = (
    analysis_df
    .groupBy("cust_id")
    .agg(
        F.first("full_name", ignorenulls=True).alias("full_name"),
        F.first("Gender", ignorenulls=True).alias("Gender"),
        F.first("age", ignorenulls=True).alias("age"),
        F.first("Region", ignorenulls=True).alias("Region"),
        F.first("State", ignorenulls=True).alias("State"),
        F.first("City", ignorenulls=True).alias("City")
    )
)

dashboard_customers_df = (
    rfm
    .join(
        customer_profile,
        on="cust_id",
        how="left"
    )
    .withColumn(
        "Last_Purchase_Year",
        F.year("last_purchase_date")
    )
    .withColumn(
        "Age_Group",
        F.when(F.col("age") < 25, "18–24")
        .when(F.col("age") < 35, "25–34")
        .when(F.col("age") < 45, "35–44")
        .when(F.col("age") < 55, "45–54")
        .otherwise("55+")
    )
    .select(
        "cust_id",
        "full_name",
        "Gender",
        "age",
        "Age_Group",
        "Region",
        "State",
        "City",
        "last_purchase_date",
        "Last_Purchase_Year",
        "recency",
        "frequency",
        "monetary",
        "R_Score",
        "F_Score",
        "M_Score",
        "RFM_Score",
        "Segment"
    )
)

print(
    f"Customer dashboard rows: "
    f"{dashboard_customers_df.count():,}"
)

display(
    dashboard_customers_df
    .orderBy("cust_id")
    .limit(20)
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Customer dashboard rows: 29,695


cust_id,full_name,Gender,age,Age_Group,Region,State,City,last_purchase_date,Last_Purchase_Year,recency,frequency,monetary,R_Score,F_Score,M_Score,RFM_Score,Segment
4,"Doughty, Reggie",M,33,25–34,Midwest,MO,Springfield,2021-09-14,2021,16,13,21079.95,5,5,5,555,Champions
15,"Diebold, Debbie",F,69,55+,Midwest,IN,Schererville,2021-02-11,2021,231,3,216.8,3,5,3,353,Loyal Customers
20,"Pulver, Eddy",M,27,25–34,South,MS,Moss,2021-04-01,2021,182,5,22379.7,4,5,5,455,Champions
21,"Kan, Adam",M,34,25–34,South,MD,Hyattsville,2021-02-03,2021,239,1,105.0,3,1,2,312,At Risk
28,"Drain, Reinaldo",M,71,55+,West,WA,Woodland,2020-10-17,2020,348,1,70.0,1,1,1,111,Lost Customers
32,"Horne, Reginald",M,62,55+,West,AK,Shaktoolik,2021-05-28,2021,125,62,44030.98,5,5,5,555,Champions
33,"Rapoza, Darnell",F,69,55+,West,AZ,Bullhead City,2021-05-26,2021,127,36,29248.53,5,5,5,555,Champions
41,"Batty, Angelo",M,23,18–24,South,DC,Washington,2020-11-18,2020,316,1,219.9,1,1,3,113,Lost Customers
44,"Ro, Kendall",F,35,35–44,South,AR,Sulphur Rock,2021-04-01,2021,182,2,2237.02,4,4,4,444,Champions
47,"Schock, Jessie",M,33,25–34,Northeast,PA,Landenberg,2020-12-31,2020,273,9,12425.01,3,5,5,355,Loyal Customers


### Create the transaction-level dashboard dataset: dashboard_transactions_df

In [0]:
dashboard_transactions_df = (
    analysis_df
    .join(
        rfm.select(
            "cust_id",
            "R_Score",
            "F_Score",
            "M_Score",
            "RFM_Score",
            "Segment"
        ),
        on="cust_id",
        how="left"
    )
    .withColumn(
        "Year",
        F.year("order_date")
    )
    .withColumn(
        "Quarter",
        F.quarter("order_date")
    )
    .withColumn(
        "Month_Number",
        F.month("order_date")
    )
    .withColumn(
        "Month_Name",
        F.date_format("order_date", "MMMM")
    )
    .withColumn(
        "Month_Start",
        F.trunc("order_date", "month")
    )
    .select(
        "order_id",
        "order_date",
        "Year",
        "Quarter",
        "Month_Number",
        "Month_Name",
        "Month_Start",
        "cust_id",
        "full_name",
        "Gender",
        "age",
        "Region",
        "State",
        "City",
        "category",
        "sku",
        "qty_ordered",
        "price",
        "discount_amount",
        "total",
        "payment_method",
        "status_clean",
        "R_Score",
        "F_Score",
        "M_Score",
        "RFM_Score",
        "Segment"
    )
)

print(
    f"Transaction dashboard rows: "
    f"{dashboard_transactions_df.count():,}"
)

display(
    dashboard_transactions_df
    .orderBy("order_date", "order_id")
    .limit(20)
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Transaction dashboard rows: 85,518


order_id,order_date,Year,Quarter,Month_Number,Month_Name,Month_Start,cust_id,full_name,Gender,age,Region,State,City,category,sku,qty_ordered,price,discount_amount,total,payment_method,status_clean,R_Score,F_Score,M_Score,RFM_Score,Segment
100354680,2020-10-01,2020,4,10,October,2020-10-01,60124,"Titus, Jani",F,43,South,OK,Vinson,Men's Fashion,mdeal_DMC-610-8,9,149.9,0.0,1199.2,cod,complete,1,1,4,114,Lost Customers
100354680,2020-10-01,2020,4,10,October,2020-10-01,60124,"Titus, Jani",F,43,South,OK,Vinson,Men's Fashion,oasis_Oasis-061-36,9,79.9,0.0,639.2,cod,complete,1,1,4,114,Lost Customers
100354689,2020-10-01,2020,4,10,October,2020-10-01,56449,"Raminez, Elizbeth",F,71,Midwest,IL,Brownstown,Men's Fashion,Head_811506-L,3,265.0,0.0,530.0,Payaxis,complete,1,4,4,144,At Risk
100354689,2020-10-01,2020,4,10,October,2020-10-01,56449,"Raminez, Elizbeth",F,71,Midwest,IL,Brownstown,Men's Fashion,head_287006-NV,2,129.0,0.0,129.0,Payaxis,complete,1,4,4,144,At Risk
100354691,2020-10-01,2020,4,10,October,2020-10-01,56449,"Raminez, Elizbeth",F,71,Midwest,IL,Brownstown,Men's Fashion,mytee_myteeL125,4,115.0,0.0,345.0,Payaxis,complete,1,4,4,144,At Risk
100354699,2020-10-01,2020,4,10,October,2020-10-01,60131,"Waymire, Abe",M,71,South,AR,Fort Smith,Women's Fashion,memsaab_E-H-U-Skin-38B,2,175.0,0.0,175.0,cod,complete,1,1,2,112,Lost Customers
100354701,2020-10-01,2020,4,10,October,2020-10-01,52163,"Stough, Florentino",M,60,Northeast,NJ,Riverton,Appliances,Fantastic_FT-1153,4,34.5,0.0,103.5,cod,complete,1,1,2,112,Lost Customers
100354704,2020-10-01,2020,4,10,October,2020-10-01,60132,"Criss, Harrison",M,75,South,WV,Huntington,Women's Fashion,sitara_210-B,2,121.5,0.0,121.5,Payaxis,complete,1,1,3,113,Lost Customers
100354704,2020-10-01,2020,4,10,October,2020-10-01,60132,"Criss, Harrison",M,75,South,WV,Huntington,Women's Fashion,sitara_205-A,2,115.5,0.0,115.5,Payaxis,complete,1,1,3,113,Lost Customers
100354717,2020-10-01,2020,4,10,October,2020-10-01,60135,"Crawley, Antione",M,60,Midwest,ND,Linton,Women's Fashion,GA_CL-213-A-134848,2,234.8,0.0,234.8,cod,complete,1,4,3,143,At Risk


> ### Calculate the main dashboard KPIs: dashboard_kpis_df

In [0]:
transaction_kpis = (
    dashboard_transactions_df
    .agg(
        F.countDistinct("cust_id").alias("Total_Customers"),
        F.countDistinct("order_id").alias("Total_Orders"),
        F.round(F.sum("total"), 2).alias("Total_Revenue")
    )
)

customer_kpis = (
    dashboard_customers_df
    .agg(
        F.round(
            F.avg("monetary"),
            2
        ).alias("Average_Customer_Spend")
    )
)

dashboard_kpis_df = transaction_kpis.crossJoin(customer_kpis)

display(dashboard_kpis_df)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Total_Customers,Total_Orders,Total_Revenue,Average_Customer_Spend
29695,61258,6.284565818E7,2116.37


### Prepare the monthly sales trend: monthly_sales_summary

In [0]:
monthly_sales_summary = (
    dashboard_transactions_df
    .groupBy(
        "Month_Start",
        "Year",
        "Segment"
    )
    .agg(
        F.round(
            F.sum("total"),
            2
        ).alias("Monthly_Revenue"),
        F.countDistinct(
            "order_id"
        ).alias("Monthly_Orders"),
        F.countDistinct(
            "cust_id"
        ).alias("Monthly_Customers")
    )
    .orderBy(
        "Month_Start",
        "Segment"
    )
)

display(monthly_sales_summary)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Month_Start,Year,Segment,Monthly_Revenue,Monthly_Orders,Monthly_Customers
2020-10-01,2020,At Risk,234564.92,530,251
2020-10-01,2020,Champions,206078.96,284,168
2020-10-01,2020,Lost Customers,319932.34,875,875
2020-10-01,2020,Loyal Customers,51540.49,166,133
2020-11-01,2020,At Risk,765512.57,946,484
2020-11-01,2020,Champions,789600.17,789,355
2020-11-01,2020,Lost Customers,956511.42,1362,1362
2020-11-01,2020,Loyal Customers,190568.98,341,232
2020-12-01,2020,At Risk,1.564714191E7,14386,7902
2020-12-01,2020,Champions,6958811.3,5131,1215


### Prepare the regional overview: regional_summary

In [0]:
regional_summary = (
    dashboard_transactions_df
    .groupBy(
        "Region",
        "Year",
        "Segment"
    )
    .agg(
        F.round(
            F.sum("total"),
            2
        ).alias("Regional_Revenue"),
        F.countDistinct(
            "order_id"
        ).alias("Regional_Orders"),
        F.countDistinct(
            "cust_id"
        ).alias("Regional_Customers")
    )
    .orderBy(
        F.desc("Regional_Revenue")
    )
)

display(regional_summary)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Region,Year,Segment,Regional_Revenue,Regional_Orders,Regional_Customers
South,2021,Champions,7879885.32,5118,1318
South,2020,At Risk,5924869.06,5685,3000
Midwest,2021,Champions,5808324.12,4125,983
Midwest,2020,At Risk,4467050.31,4459,2289
West,2021,Champions,4032320.45,2678,683
Northeast,2021,Champions,3501765.09,2447,639
West,2020,At Risk,3167987.59,2801,1469
Northeast,2020,At Risk,3087312.44,2917,1511
South,2020,Champions,2828761.74,2172,465
South,2020,Loyal Customers,2814133.46,2282,799


### Save the two dashboard datasets

In [0]:
CUSTOMER_DASHBOARD_TABLE = (
    "bia_5203.default.rfm_customer_dashboard"
)

TRANSACTION_DASHBOARD_TABLE = (
    "bia_5203.default.rfm_transaction_dashboard"
)

(
    dashboard_customers_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(CUSTOMER_DASHBOARD_TABLE)
)

(
    dashboard_transactions_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TRANSACTION_DASHBOARD_TABLE)
)

print("Dashboard tables saved successfully.")
print(f"Customer table: {CUSTOMER_DASHBOARD_TABLE}")
print(f"Transaction table: {TRANSACTION_DASHBOARD_TABLE}")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Dashboard tables saved successfully.
Customer table: bia_5203.default.rfm_customer_dashboard
Transaction table: bia_5203.default.rfm_transaction_dashboard


### Verify the saved dashboard tables

In [0]:
saved_customer_df = spark.table(
    CUSTOMER_DASHBOARD_TABLE
)

saved_transaction_df = spark.table(
    TRANSACTION_DASHBOARD_TABLE
)

print(
    f"Saved customer rows: "
    f"{saved_customer_df.count():,}"
)

print(
    f"Saved distinct customers: "
    f"{saved_customer_df.select('cust_id').distinct().count():,}"
)

print(
    f"Saved transaction rows: "
    f"{saved_transaction_df.count():,}"
)

print(
    f"Saved distinct orders: "
    f"{saved_transaction_df.select('order_id').distinct().count():,}"
)

display(
    saved_customer_df.limit(5)
)

display(
    saved_transaction_df.limit(5)
)

Saved customer rows: 29,695
Saved distinct customers: 29,695
Saved transaction rows: 85,518
Saved distinct orders: 61,258


cust_id,full_name,Gender,age,Age_Group,Region,State,City,last_purchase_date,Last_Purchase_Year,recency,frequency,monetary,R_Score,F_Score,M_Score,RFM_Score,Segment
54593,"Stairs, Kandis",F,34,25–34,South,NC,Autryville,2021-06-06,2021,116,1,0.1,5,1,1,511,Potential Loyalists
70044,"Albertson, Hilton",M,25,25–34,South,TN,Arrington,2020-12-21,2020,283,1,2.7,2,2,1,221,At Risk
41410,"Till, Jay",F,40,35–44,Northeast,PA,Tiona,2020-10-01,2020,364,1,3.8,1,1,1,111,Lost Customers
61240,"Bergquist, Thad",M,70,55+,Midwest,IN,Coal City,2020-10-21,2020,344,1,5.0,1,1,1,111,Lost Customers
86773,"Burgoon, Eldora",F,42,35–44,Midwest,IA,Lisbon,2021-03-12,2021,202,1,5.9,4,3,1,431,Loyal Customers


order_id,order_date,Year,Quarter,Month_Number,Month_Name,Month_Start,cust_id,full_name,Gender,age,Region,State,City,category,sku,qty_ordered,price,discount_amount,total,payment_method,status_clean,R_Score,F_Score,M_Score,RFM_Score,Segment
100354680,2020-10-01,2020,4,10,October,2020-10-01,60124,"Titus, Jani",F,43,South,OK,Vinson,Men's Fashion,mdeal_DMC-610-8,9,149.9,0.0,1199.2,cod,complete,1,1,4,114,Lost Customers
100354680,2020-10-01,2020,4,10,October,2020-10-01,60124,"Titus, Jani",F,43,South,OK,Vinson,Men's Fashion,oasis_Oasis-061-36,9,79.9,0.0,639.2,cod,complete,1,1,4,114,Lost Customers
100403536,2020-12-24,2020,4,12,December,2020-12-01,42485,"Eaker, Lee",M,28,South,FL,Graham,Appliances,APPWES59CA4D59CB41A,3,254.8,101.92,407.68,Easypay,complete,3,4,4,344,Loyal Customers
100403536,2020-12-24,2020,4,12,December,2020-12-01,42485,"Eaker, Lee",M,28,South,FL,Graham,Appliances,APPPHI5A09939191938,3,315.5,126.2,504.8,Easypay,complete,3,4,4,344,Loyal Customers
100403536,2020-12-24,2020,4,12,December,2020-12-01,42485,"Eaker, Lee",M,28,South,FL,Graham,Home & Living,HALSHO59F836EF959B9,3,69.8,27.92,111.68,Easypay,complete,3,4,4,344,Loyal Customers
